In [2]:
import pandas as pd
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    GenerativeModel,
    Tool,
    HarmCategory,
    HarmBlockThreshold
)
import json
import re
import pickle


In [ ]:
PROJECT_ID = "proj-sales-recommender-dev"
LOCATION = "us-central1" 

import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION)


In [ ]:
MODEL_ID = "gemini-2.0-flash-001"

model = GenerativeModel(
    MODEL_ID,
    safety_settings={
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        },

)

In [ ]:
def get_candidate_filters():

    with open('data/boolean_filters_latest.json') as f:  # Use 'r' for reading
        product_filter_json = json.load(f)  ## Load the JSON data
    
    materials_df = pd.read_csv('data/relevant_materials.csv')
    materials = materials_df.to_json(orient='records')

    return product_filter_json, materials

In [ ]:
def filter_by_stage(cc_data_df ):
    stage_filter = {"high": ["Biddate Set", "Construction Documents", "General Contractor Award", "Low Bids Announced", "SUBBIDS: ASAP"], "moderate": ["Construction Underway", "Post Bid"], "ignore": ["Cancelled", "Design Development", "Disqualified Lead", "Duplicate Project", "Pre-Design", "Schematic Design"]}
    stages_to_include = stage_filter["high"] + stage_filter["moderate"]
    filtered_by_stage = cc_data_df[cc_data_df['Stage'].isin(stages_to_include)]
    return filtered_by_stage

In [ ]:
def filter_by_project_value(cc_data_df):
    if cc_data_df['Valuation_Value'].dtype == 'object':  
        cc_data_df.loc[:, 'Valuation_Value'] = pd.to_numeric(cc_data_df.loc[:, 'Valuation_Value'], errors='coerce')
    filtered_cc_value = cc_data_df[cc_data_df['Valuation_Value'] > 1000000]
    return filtered_cc_value


In [ ]:
def get_cc_data(pkl_path):
    construct_connect_data = pd.read_pickle(pkl_path)
    return construct_connect_data


In [13]:
def run_prompt_boolean_filter(product, search_query, cc_project_json):
    question = f''' 
    **Objective:** Identify if the ConstructConnect project is related to the given product.

    **Instructions:**
        1. **Analyze the provided JSON data** representing ConstructConnect project to understand relevant fields and data.
        2. **Utilize the Boolean filters** corresponding to the **Product Category** to identify if the project is related to the given product by matching across multpile project fields.
        3. **Respond as YES or NO with a reason for your answer in a valid JSON as provided in the Example Output below. RETURN ONLY THE JSON RESPONSE.**

    **JSON Project Data:** 
    {cc_project_json}

    **Product Category:**
    {product}
   
    **Boolean Filters:**
    {search_query}

    **Example Output:**
    {{"ProjectID": 1006193703, "Product": {product}, "Project related to Product": "YES", "Product Reasoning": reason for relation response}}
    '''

    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)
    return response.text


In [ ]:
def get_project_product_relation(product_filter_json, cc_data):
    all_rows = []
    eligible_for_material_filter = pd.DataFrame()  # Initialize an empty DataFrame


    for index, row in cc_data.iterrows():  # Iterate through rows
        row_json_str = row.to_json()  # Convert row to dictionary
        cc_project_json_record = json.loads(row_json_str) 
        for item in product_filter_json:
            output = run_prompt_boolean_filter(item['Filter'], item['Query'], cc_project_json_record)
            otpt = output.replace("```json", "").replace("```", "")
            output_json = json.loads(otpt)
            if (output_json["Project related to Product"]) == "YES":
                new_row = row.to_dict()  # Convert row to dictionary for easier appending
                new_row['PRODUCT'] = item['Filter']
                new_row['Query'] = item['Query']
                new_row['PRODUCT_PRIORITY'] = item['Priority']
                all_rows.append(new_row)  # Append the modified row
                
                # Concatenate the item DataFrame with the original row
    if all_rows: # Check if any rows were added to prevent errors if the list is empty
        eligible_for_material_filter = pd.DataFrame(all_rows)  # Create DataFrame from the list of dictionaries
    return eligible_for_material_filter

In [15]:
def run_prompt_material_filter(materials, cc_project_json):
    question = f''' 
    **Objective:** Identify if the ConstructConnect project has any of the given relevant materials.

    **Instructions:**
        1. **Analyze the provided JSON data** representing ConstructConnect project to understand relevant fields and data.
        2. **Utilize the Materials list** to identify if the project has any of the materials listed by matching across multpile project fields.
        3. **Respond as YES or NO with a list of all the materials mentioned in your answer in a valid JSON as provided in the Example Output below. RETURN ONLY THE JSON RESPONSE.**

    **JSON Project Data:** 
    {cc_project_json}

    **Materials List:**
    {materials}
   

    **Example Output:**
    {{"ProjectID": 1006193703, "Relevant materials present": YES, "Materials": ["Material1","Material2", ...], "Reasoning": reason for relation response}}
    '''

    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)
    return response.text

In [ ]:
def filter_projects_by_material(materials, eligible_for_material_filter):

    eligible_for_classification = pd.DataFrame()  # Initialize an empty DataFrame

    for index, row in eligible_for_material_filter.iterrows():  # Iterate through rows
        row_json_str = row.to_json()  # Convert row to dictionary
        cc_project_json_record = json.loads(row_json_str) 
        
        output = run_prompt_material_filter(materials, cc_project_json_record)
        otpt = output.replace("```json", "").replace("```", "")
        try:
            output_json = json.loads(otpt)
            if (output_json["Relevant materials present"]) == "YES":
                eligible_for_classification = pd.concat([eligible_for_classification, row.to_frame().T], ignore_index=True) # Append the row
        except:
            print("Error parsing output: ",otpt)
    return eligible_for_classification
            


In [ ]:
def process_dataframe(data_df, filtered_flag):
    
    if not filtered_flag:
        cc_data_filtered_stage = filter_by_stage(data_df)
        cc_data_filtered = filter_by_project_value(cc_data_filtered_stage)
    else:
        cc_data_filtered = data_df
    product_filter_json, materials_json = get_candidate_filters()
    candidates_for_material_filter = get_project_product_relation(product_filter_json, cc_data_filtered)
    candidates_for_classification = filter_projects_by_material(materials_json, candidates_for_material_filter)
    return candidates_for_classification

In [ ]:
cc_data = get_cc_data('data/construct_connect_data.pkl')
df_to_classify = process_dataframe(cc_data, False)


In [ ]:
data_pkl_path = 'data/eligible_for_classification_60.pkl'
with open(data_pkl_path, 'wb') as file:
        pickle.dump(df_to_classify, file)